In [47]:
from Auxiliaries import evaluate_policy
from Policies import ProBSP
from NewEnvironment import Inventory, GeometricOrderPipeline, InventoryRS
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib import MaskablePPO as PPO
from MaskedDQN import MaskedDoubleDQN as DDQN

This file provides an example on defining an environment and learning a DQN, and PPO policies.
The inventory problem has the following parameters:
M=2, p=0.33, B=3, MTTF=10, Co=2, Ce=5


In [48]:
num_machines = 2
lead_times_p = 1/2
max_batch_size = 3
mttf = 10
sort_degradation = True  # Sort degradation to benefit from the reduction in state space

order_pipeline = GeometricOrderPipeline(num_machines, lead_times_p)
inventory = Inventory(machines=num_machines,
                      order_pipeline=order_pipeline,
                      mttf=mttf,
                      sorted_degradation=sort_degradation)

n, xo = 0, 60
probsp = ProBSP(env=inventory, n=n, xo=xo, max_batch_size=max_batch_size)

# Evaluate the ProBSP

In [49]:
evaluate_policy(env=inventory, policy=probsp, replication=8, processors=4)

ProBSP with N=0 and Xo=60.0 - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8615403845279761 ± 0.0052 	 E[S]=0.2567 ± 0.0017 	 FR=0.7461 ± 0.0052 	 Total Eval Time=3.3864


(np.float64(0.8615403845279761),
 np.float64(0.0051946114359146325),
 np.float64(0.7460746369779327),
 np.float64(0.00523035579729058),
 np.float64(0.2566590909090912),
 np.float64(0.0016657711092555908))

# Learning using a PPO policy

In [50]:
print("Learning a policy using PPO")
ppo = PPO(MaskableActorCriticPolicy, env=inventory, verbose=0)
ppo.learn(800000)
print("Finished Learning PPO policy, evaluating .......")
evaluate_policy(env=inventory, policy=ppo, replication=8, processors=4)

Learning a policy using PPO
Finished Learning PPO policy, evaluating .......
<sb3_contrib.ppo_mask.ppo_mask.MaskablePPO object at 0x168dc1180> - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.818746875142039 ± 0.0071 	 E[S]=0.2839 ± 0.0064 	 FR=0.7002 ± 0.0086 	 Total Eval Time=41.3591


(np.float64(0.818746875142039),
 np.float64(0.007078183951909987),
 np.float64(0.7001936949570338),
 np.float64(0.008620168632543307),
 np.float64(0.28391477272727295),
 np.float64(0.006409511490873999))

# Learning using a DQN policy

In [51]:
print("Learning a policy using DQN")
dqn = DDQN(env=inventory)
dqn.learn(800000)
print("Finished Learning DQN policy, evaluating .......")
evaluate_policy(env=inventory, policy=dqn, replication=8, processors=4)

Learning a policy using DQN
Starting Learning for 800,000 steps
Finished Learning in 801.66s
Finished Learning DQN policy, evaluating .......
DDQN - Inventory - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8158208717785556 ± 0.0048 	 E[S]=0.2497 ± 0.0036 	 FR=0.6741 ± 0.0044 	 Total Eval Time=16.3826


(np.float64(0.8158208717785556),
 np.float64(0.004809778576282698),
 np.float64(0.6741453559457729),
 np.float64(0.004389876121595564),
 np.float64(0.24972727272727244),
 np.float64(0.0035808258893143953))

# Inventory with Reward Shaping using the ProBSP

In [52]:
# When reward shaping is included using the ProBSP
inventory_rs = InventoryRS(machines=num_machines,
                           order_pipeline=order_pipeline,
		                   mttf=mttf,
		                   sorted_degradation=sort_degradation,
                           probsp=True,
                           bsp=False,
                           probsp_xo=xo,
                           probsp_n=n,
                           gamma=0.99
                           )

In [53]:
# Again, let us learn using a PPO policy
print("Learning a policy using PPO")
ppo = PPO(MaskableActorCriticPolicy, env=inventory_rs, verbose=0)
ppo.learn(800000)
print("Finished Learning PPO policy, evaluating .......")
evaluate_policy(env=inventory_rs, policy=ppo, replication=8, processors=4)


Learning a policy using PPO
Finished Learning PPO policy, evaluating .......
<sb3_contrib.ppo_mask.ppo_mask.MaskablePPO object at 0x168299ba0> - Inventory-RS - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8209285941548111 ± 0.0072 	 E[S]=0.2826 ± 0.0045 	 FR=0.7023 ± 0.0085 	 Total Eval Time=40.3910


(np.float64(0.8209285941548111),
 np.float64(0.00717205523986255),
 np.float64(0.7022539106166305),
 np.float64(0.00850507470365638),
 np.float64(0.2825965909090916),
 np.float64(0.004544513396884233))

In [54]:
# Let us learn using a DQN policy
print("Learning a policy using DQN")
dqn = DDQN(env=inventory_rs)
dqn.learn(800000)
print("Finished Learning DQN policy, evaluating .......")
evaluate_policy(env=inventory_rs, policy=dqn, replication=8, processors=4)

Learning a policy using DQN
Starting Learning for 800,000 steps
Finished Learning in 787.94s
Finished Learning DQN policy, evaluating .......
DDQN - Inventory-RS - 8 replications - 20000 steps - 2000 burn-in period avg:
Costs=0.8152640789055043 ± 0.0091 	 E[S]=0.2738 ± 0.0032 	 FR=0.7031 ± 0.0095 	 Total Eval Time=21.7026


(np.float64(0.8152640789055043),
 np.float64(0.009089954346739535),
 np.float64(0.7030551532694894),
 np.float64(0.009516039133847935),
 np.float64(0.2737954545454542),
 np.float64(0.003180600416314607))